# LeetCode 396 — Rotate Function bằng Python

Link đề: https://leetcode.com/problems/rotate-function/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def maxRotateFunction(self, nums: List[int]) -> int:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; các ví dụ đều được chạy lại bằng code trong notebook.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xoay mảng `k` bước theo chiều kim đồng hồ, tính "hàm xoay" `F(k) = 0*arr[0] + 1*arr[1] + ... + (n-1)*arr[n-1]` rồi lấy **giá trị lớn nhất** trong `F(0), ..., F(n-1)`.
2. Nắm công cụ Python: `enumerate`, `sum` với generator, cắt lát `nums[-k:] + nums[:-k]`, phép `%` để "quay vòng" chỉ số, `itertools.accumulate` cho tổng tiền tố.
3. Đi từ cách **dựng từng mảng xoay `O(n²)`** → **bỏ dựng mảng bằng công thức chỉ số** → **công thức truy hồi `F(k) = F(k-1) + S - n*nums[n-k]` chạy `O(n)`** → **công thức đóng dùng tổng hậu tố**.
4. Hiểu mấu chốt: khi xoay một bước, **mọi phần tử tăng hệ số thêm 1**, riêng phần tử cuối nhảy từ hệ số `n-1` về `0`; do đó hiệu `F(k) - F(k-1)` có dạng đóng, không cần tính lại từ đầu.
5. Tránh các lỗi kinh điển: khởi tạo `ans = 0` (sai khi đáp án âm), dùng nhầm chỉ số `nums[i]` thay vì `nums[n-i]`, quên cộng dồn `f`, nhầm hệ số `n` với `n-1`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` độ dài `n`. Gọi `arr_k` là mảng thu được khi **xoay `nums` đi `k` vị trí theo chiều kim đồng hồ** (phần tử cuối nhảy lên đầu mỗi lần xoay). Định nghĩa **hàm xoay**:

`F(k) = 0 * arr_k[0] + 1 * arr_k[1] + ... + (n - 1) * arr_k[n - 1]`

Hãy trả về **giá trị lớn nhất** trong các giá trị `F(0), F(1), ..., F(n-1)`. Dữ liệu đảm bảo đáp án vừa trong số nguyên 32 bit.

**Ví dụ 1:** `nums = [4,3,2,6]` → `26`. Thật vậy: `F(0) = 0+3+4+18 = 25`, `F(1) = 0+4+6+6 = 16`, `F(2) = 0+6+8+9 = 23`, `F(3) = 0+2+12+12 = 26`; lớn nhất là `F(3) = 26`.

**Ví dụ 2:** `nums = [100]` → `0` (chỉ có `F(0) = 0*100 = 0`).

**Ràng buộc:** `n == nums.length`, `1 <= n <= 10^5`, `-100 <= nums[i] <= 100`.

In [1]:
from itertools import accumulate
import random
import time
from typing import List

examples = [
    ([4, 3, 2, 6], 26),
    ([100], 0),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[4, 3, 2, 6] -> kỳ vọng 26
[100] -> kỳ vọng 0


### 2.1. Đọc kỹ đề: "chiều kim đồng hồ" có quan trọng không?

Xoay **thuận** hay **ngược** chiều kim đồng hồ chỉ làm **thứ tự liệt kê** các mảng `arr_k` khác đi; **tập các mảng** có được sau `n` lần xoay là như nhau. Vì ta lấy **max** trên cả tập nên chiều xoay không ảnh hưởng đáp án. Tuy vậy, khi dựng công thức truy hồi ta phải bám đúng một chiều (ở đây: thuận, phần tử cuối nhảy lên đầu), nếu không sẽ nhầm `nums[n-k]` với `nums[k-1]`.

Điểm cần nhớ: `n` có thể lên tới `10^5`, mỗi `nums[i]` chỉ trong `[-100, 100]`, và `F(k)` có thể **âm** (ví dụ `nums = [-1, -2]`). Hai nhận xét này quyết định: (1) cần thuật toán gần tuyến tính, (2) không được khởi tạo đáp án bằng `0`.

In [2]:
nums = [4, 3, 2, 6]
n = len(nums)
clockwise = {tuple(nums[n - k:] + nums[:n - k]) for k in range(n)}
counter = {tuple(nums[k:] + nums[:k]) for k in range(n)}
print(clockwise == counter)          # True: cùng một tập mảng, chỉ khác thứ tự liệt kê

True


## 3. Kiến thức nền

### 3.1. `enumerate` và `sum` với generator

`enumerate(arr)` cho ta cặp `(chỉ số, giá trị)`. Kết hợp với `sum(... for ... in ...)` ta viết gọn tổng `0*arr[0] + 1*arr[1] + ...`.

In [3]:
arr = [4, 3, 2, 6]
for i, v in enumerate(arr):
    print(f"i={i}, v={v}, i*v={i * v}")
print("tổng:", sum(i * v for i, v in enumerate(arr)))    # 0 + 3 + 4 + 18 = 25

i=0, v=4, i*v=0
i=1, v=3, i*v=3
i=2, v=2, i*v=4
i=3, v=6, i*v=18
tổng: 25


### 3.2. Xoay mảng bằng cắt lát (slicing)

"Xoay theo chiều kim đồng hồ `k` bước" nghĩa là lấy `k` phần tử **cuối** đưa lên **đầu**. Trong Python: `nums[-k:] + nums[:-k]` (cẩn thận khi `k = 0`, vì `nums[-0:]` là **toàn bộ** mảng!). Cách an toàn: `nums[n-k:] + nums[:n-k]`.

In [4]:
nums = [4, 3, 2, 6]
n = len(nums)
for k in range(n):
    print(f"k={k}:", nums[n - k:] + nums[:n - k])

# cạm bẫy khi k = 0:
print("nums[-0:] =", nums[-0:], "(là cả mảng, không phải rỗng)")

k=0: [4, 3, 2, 6]
k=1: [6, 4, 3, 2]
k=2: [2, 6, 4, 3]
k=3: [3, 2, 6, 4]
nums[-0:] = [4, 3, 2, 6] (là cả mảng, không phải rỗng)


### 3.3. Phép `%` để quay vòng chỉ số

Sau khi xoay `k` bước, phần tử ở vị trí `i` của `arr_k` chính là `nums[(i - k) % n]`. Python cho kết quả `%` luôn **không âm** khi số chia dương, nên `(i - k) % n` an toàn cả khi `i - k < 0` (ở nhiều ngôn ngữ khác cần cộng thêm `n`).

In [5]:
n = 4
print([(-1) % n, (-2) % n, (-3) % n])           # [3, 2, 1]
nums = [4, 3, 2, 6]
k = 1
print([nums[(i - k) % n] for i in range(n)])    # [6, 4, 3, 2] đúng bằng arr_1

[3, 2, 1]
[6, 4, 3, 2]


### 3.4. Tổng tiền tố/hậu tố với `accumulate`

`itertools.accumulate(arr)` trả về các tổng cộng dồn: `[a0, a0+a1, a0+a1+a2, ...]`. Từ đó, tổng của `k` phần tử cuối mảng = tổng toàn bộ trừ tổng tiền tố `n-k` phần tử đầu. Ta sẽ cần ý này ở Cách 4.

In [6]:
nums = [4, 3, 2, 6]
prefix = [0] + list(accumulate(nums))           # prefix[j] = tổng j phần tử đầu
print(prefix)                                   # [0, 4, 7, 9, 15]
n = len(nums)
for k in range(n + 1):
    print(f"tổng {k} phần tử cuối =", prefix[n] - prefix[n - k], "| kiểm tra:", sum(nums[n - k:]))

[0, 4, 7, 9, 15]
tổng 0 phần tử cuối = 0 | kiểm tra: 0
tổng 1 phần tử cuối = 6 | kiểm tra: 6
tổng 2 phần tử cuối = 8 | kiểm tra: 8
tổng 3 phần tử cuối = 11 | kiểm tra: 11
tổng 4 phần tử cuối = 15 | kiểm tra: 15


## 4. Ý tưởng nền tảng: tính tay một ví dụ

Với `nums = [4,3,2,6]`, ta xoay từng bước và tính `F`. Đây chính là đề; nhưng hãy chú ý **mỗi lần xoay, các hệ số thay đổi thế nào**:

- `arr_0 = [4,3,2,6]` có hệ số `0,1,2,3` → `F(0) = 25`.
- `arr_1 = [6,4,3,2]`: số `6` từ hệ số `3` về `0`; `4` từ `0` lên `1`; `3` từ `1` lên `2`; `2` từ `2` lên `3`.
- Mỗi phần tử **không phải phần tử cuối** được tăng hệ số thêm `1`, phần tử cuối **mất `n-1 = 3` hệ số**.

Ta đang "đi tắt" tới công thức ở mục 6. Trước hết cứ làm thật chậm và thật rõ.

In [7]:
nums = [4, 3, 2, 6]
n = len(nums)
for k in range(n):
    arr = nums[n - k:] + nums[:n - k]
    terms = " + ".join(f"{i}*{v}" for i, v in enumerate(arr))
    print(f"k={k}: arr={arr}  F = {terms} = {sum(i * v for i, v in enumerate(arr))}")

k=0: arr=[4, 3, 2, 6]  F = 0*4 + 1*3 + 2*2 + 3*6 = 25
k=1: arr=[6, 4, 3, 2]  F = 0*6 + 1*4 + 2*3 + 3*2 = 16
k=2: arr=[2, 6, 4, 3]  F = 0*2 + 1*6 + 2*4 + 3*3 = 23
k=3: arr=[3, 2, 6, 4]  F = 0*3 + 1*2 + 2*6 + 3*4 = 26


Cách nhìn "theo từng phần tử": hãy theo dõi **hệ số** (vị trí) của từng giá trị gốc qua các lần xoay. Mỗi hàng là một giá trị của `nums`, mỗi cột là một `k`:

- giá trị `4` (ban đầu ở vị trí 0): hệ số `0, 1, 2, 3` (tăng dần, đến `k=3` thì vị trí cuối);
- giá trị `6` (ban đầu ở vị trí 3): hệ số `3, 0, 1, 2` (rơi từ `3` xuống `0` rồi lại tăng).

Quy luật: **hệ số của mỗi giá trị tăng `1` sau mỗi lần xoay, và khi vượt `n-1` thì quay về `0`.** Đây là mấu chốt cho mục 6.

In [8]:
nums = [4, 3, 2, 6]
n = len(nums)
print("giá trị | hệ số ở k=0,1,2,3")
for j, v in enumerate(nums):
    coeffs = [(j + k) % n for k in range(n)]          # vị trí của nums[j] trong arr_k là (j + k) % n
    print(f"{v:>7} | {coeffs}")

giá trị | hệ số ở k=0,1,2,3
      4 | [0, 1, 2, 3]
      3 | [1, 2, 3, 0]
      2 | [2, 3, 0, 1]
      6 | [3, 0, 1, 2]


## 5. Cách 1 — Brute Force: dựng từng mảng xoay

**Ý tưởng:** làm đúng theo đề. Với mỗi `k = 0..n-1`: dựng `arr_k` bằng cắt lát, tính `F(k)` bằng một vòng lặp, giữ giá trị lớn nhất.

**Chú ý:** phải khởi tạo `best` bằng `F(0)` hoặc `-infinity`, **không** phải `0` (đáp án có thể âm).

In [9]:
def max_rotate_brute(nums):
    n = len(nums)
    best = None
    for k in range(n):
        arr = nums[n - k:] + nums[:n - k]               # xoay k bước (k phần tử cuối lên đầu)
        f = sum(i * v for i, v in enumerate(arr))
        if best is None or f > best:
            best = f
    return best


for nums, expected in examples:
    got = max_rotate_brute(nums)
    print(nums, "->", got, "| kỳ vọng", expected)
    assert got == expected

[4, 3, 2, 6] -> 26 | kỳ vọng 26
[100] -> 0 | kỳ vọng 0


**Độ phức tạp:** `n` phép xoay, mỗi phép tốn `O(n)` để dựng mảng và `O(n)` để tính tổng ⇒ **Time `O(n²)`**, Space `O(n)` cho mảng xoay.

**Nút thắt:** với `n = 10^5` ta có khoảng `10^10` phép tính — quá chậm. Ta đang **tính lại từ đầu** mỗi lần dù hai mảng xoay liên tiếp gần như giống nhau. Hãy đếm số phép nhân để thấy mức tăng bậc hai.

In [10]:
for n in [10, 1000, 10**5]:
    print(f"n = {n:>6}: số phép nhân i*v của Cách 1 = n*n = {n * n:,}")

n =     10: số phép nhân i*v của Cách 1 = n*n = 100
n =   1000: số phép nhân i*v của Cách 1 = n*n = 1,000,000
n = 100000: số phép nhân i*v của Cách 1 = n*n = 10,000,000,000


## 6. Ý tưởng thuật toán / bước đột phá

Hai mảng xoay liên tiếp khác nhau rất ít. Đặt `S = sum(nums)`. Khi xoay từ `arr_{k-1}` sang `arr_k`:

- `arr_{k-1}` có phần tử cuối là `last = nums[n - k]` (vì `arr_{k-1}[n-1] = nums[(n-1-(k-1)) % n] = nums[n-k]`).
- Trong `F(k-1)`, `last` mang hệ số `n-1`. Trong `F(k)`, `last` nằm đầu mảng với hệ số `0` ⇒ **mất** `(n-1)*last`.
- Mỗi phần tử còn lại (tổng là `S - last`) tăng hệ số thêm `1` ⇒ **được thêm** `S - last`.

Cộng lại: `F(k) - F(k-1) = (S - last) - (n-1)*last = S - n*last`. Vậy:

**`F(k) = F(k-1) + S - n * nums[n - k]`**

Chỉ cần tính `F(0)` một lần (`O(n)`), rồi `n-1` bước cập nhật `O(1)`. Kiểm tra bằng số trước khi code.

In [11]:
nums = [4, 3, 2, 6]
n, S = len(nums), sum(nums)
F = [sum(i * v for i, v in enumerate(nums[n - k:] + nums[:n - k])) for k in range(n)]
print("F thật       :", F)
print("hiệu thật    :", [F[k] - F[k - 1] for k in range(1, n)])
print("công thức S - n*nums[n-k]:", [S - n * nums[n - k] for k in range(1, n)])

F thật       : [25, 16, 23, 26]
hiệu thật    : [-9, 7, 3]
công thức S - n*nums[n-k]: [-9, 7, 3]


Hai hàng khớp nhau (`-9, 7, 3`). Ví dụ `k = 1`: `last = nums[3] = 6`, `S = 15`, `n = 4` ⇒ `15 - 24 = -9` và `F(1) = 25 - 9 = 16` ✓.

## 7. Các cách tiếp theo

### Cách 2 — Công thức chỉ số, không dựng mảng (vẫn `O(n²)`)

**Giải thích từng đoạn:**

1. Không cần tạo `arr_k`; phần tử ở vị trí `i` của `arr_k` là `nums[(i - k) % n]` (mục 3.3).
2. `F(k) = sum(i * nums[(i - k) % n] for i in range(n))`.
3. Chỉ tiết kiệm **bộ nhớ** (Space `O(1)`), thời gian vẫn `O(n²)`.

In [12]:
def max_rotate_index(nums):
    n = len(nums)
    return max(sum(i * nums[(i - k) % n] for i in range(n)) for k in range(n))


print(max_rotate_index([4, 3, 2, 6]), max_rotate_index([100]))

26 0


In [13]:
# Code nộp LeetCode — Cách 2 (O(n^2) time, O(1) space; chạy được nhưng sẽ TLE ở n = 10^5)
class SolutionB:
    def maxRotateFunction(self, nums: List[int]) -> int:
        n = len(nums)
        return max(sum(i * nums[(i - k) % n] for i in range(n)) for k in range(n))


print(SolutionB().maxRotateFunction([4, 3, 2, 6]))

26


**Độ phức tạp Cách 2:** Time `O(n²)`, Space `O(1)`.

### Cách 3 — Công thức truy hồi (DP một chiều, `O(n)`)

**Giải thích từng đoạn:**

1. Tính `f = F(0)` và `S = sum(nums)`; đặt `ans = f`.
2. Với `k` từ `1` đến `n-1`: `f = f + S - n * nums[n - k]`, rồi `ans = max(ans, f)`.
3. Chỉ cần **một biến `f`** thay cho cả mảng DP vì `F(k)` chỉ phụ thuộc `F(k-1)`.

In [14]:
def max_rotate_recurrence(nums, verbose=False):
    n, S = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))          # F(0)
    ans = f
    if verbose:
        print(f"F(0) = {f}")
    for k in range(1, n):
        f = f + S - n * nums[n - k]                     # F(k) = F(k-1) + S - n * (phần tử cuối của arr_{k-1})
        ans = max(ans, f)
        if verbose:
            print(f"F({k}) = {f}")
    return ans


print(max_rotate_recurrence([4, 3, 2, 6], verbose=True))

F(0) = 25
F(1) = 16
F(2) = 23
F(3) = 26
26


In [15]:
# Code nộp LeetCode — Cách 3 (O(n) time, O(1) space)
class SolutionC:
    def maxRotateFunction(self, nums: List[int]) -> int:
        n, s = len(nums), sum(nums)
        f = sum(i * v for i, v in enumerate(nums))
        ans = f
        for k in range(1, n):
            f += s - n * nums[n - k]
            ans = max(ans, f)
        return ans


print(SolutionC().maxRotateFunction([4, 3, 2, 6]), SolutionC().maxRotateFunction([100]))

26 0


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(1)`.

### Cách 4 — Công thức đóng bằng tổng hậu tố

**Giải thích từng đoạn:** cộng dồn hiệu `F(j) - F(j-1) = S - n * nums[n-j]` với `j = 1..k`:

`F(k) = F(0) + k*S - n * (nums[n-1] + nums[n-2] + ... + nums[n-k])`

Tổng trong ngoặc chính là **tổng `k` phần tử cuối** (mục 3.4): `prefix[n] - prefix[n-k]`. Vậy mỗi `F(k)` tính độc lập trong `O(1)` từ mảng tổng tiền tố — tiện khi chỉ cần hỏi một vài `k` cụ thể.

In [16]:
def max_rotate_closed(nums):
    n = len(nums)
    prefix = [0] + list(accumulate(nums))
    S = prefix[n]
    f0 = sum(i * v for i, v in enumerate(nums))
    return max(f0 + k * S - n * (prefix[n] - prefix[n - k]) for k in range(n))


print(max_rotate_closed([4, 3, 2, 6]), max_rotate_closed([100]))

26 0


In [17]:
# Code nộp LeetCode — Cách 4 (O(n) time, O(n) space)
class SolutionD:
    def maxRotateFunction(self, nums: List[int]) -> int:
        n = len(nums)
        prefix = [0] + list(accumulate(nums))
        s = prefix[n]
        f0 = sum(i * v for i, v in enumerate(nums))
        return max(f0 + k * s - n * (prefix[n] - prefix[n - k]) for k in range(n))


print(SolutionD().maxRotateFunction([4, 3, 2, 6]), SolutionD().maxRotateFunction([100]))

26 0


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)` cho mảng tiền tố (có thể giảm xuống `O(1)` bằng cách cộng dồn dần, chính là Cách 3).

Lưu ý: ở Cách 4 khi `k = 0`, `prefix[n] - prefix[n - 0] = 0` nên không bị cạm bẫy `nums[-0:]` như cắt lát.

## 8. TRACE từng bước

Trace Cách 3 với `nums = [4, 3, 2, 6]` (`n = 4`, `S = 15`, `F(0) = 25`):

| k | phần tử cuối của `arr_{k-1}` = `nums[n-k]` | `S - n*nums[n-k]` | `f` mới | `ans` |
|---|---|---|---|---|
| 0 | — | — | 25 | 25 |
| 1 | `nums[3] = 6` | `15 - 24 = -9` | 16 | 25 |
| 2 | `nums[2] = 2` | `15 - 8 = 7` | 23 | 25 |
| 3 | `nums[1] = 3` | `15 - 12 = 3` | 26 | 26 |

Hàm `trace_recurrence` dưới đây in đúng các cột này, kèm `arr_k` thật để ta so sánh.

In [18]:
def trace_recurrence(nums):
    n, S = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    ans = f
    print(f"n={n}, S={S}, F(0)={f}")
    print(f"{'k':>2} | {'last=nums[n-k]':>14} | {'S-n*last':>8} | {'f mới':>6} | {'ans':>5} | arr_k thật -> F thật")
    print(f"{0:>2} | {'—':>14} | {'—':>8} | {f:>6} | {ans:>5} | {nums} -> {f}")
    for k in range(1, n):
        last = nums[n - k]
        delta = S - n * last
        f += delta
        ans = max(ans, f)
        arr = nums[n - k:] + nums[:n - k]
        real = sum(i * v for i, v in enumerate(arr))
        assert real == f
        print(f"{k:>2} | {last:>14} | {delta:>8} | {f:>6} | {ans:>5} | {arr} -> {real}")
    return ans


print("kết quả:", trace_recurrence([4, 3, 2, 6]))

n=4, S=15, F(0)=25
 k | last=nums[n-k] | S-n*last |  f mới |   ans | arr_k thật -> F thật
 0 |              — |        — |     25 |    25 | [4, 3, 2, 6] -> 25
 1 |              6 |       -9 |     16 |    25 | [6, 4, 3, 2] -> 16
 2 |              2 |        7 |     23 |    25 | [2, 6, 4, 3] -> 23
 3 |              3 |        3 |     26 |    26 | [3, 2, 6, 4] -> 26
kết quả: 26


In [19]:
# Trace thứ hai: có số âm, đáp án đạt ở F(0)
print("kết quả:", trace_recurrence([-5, 10, -3, 7, 2]))

n=5, S=11, F(0)=33
 k | last=nums[n-k] | S-n*last |  f mới |   ans | arr_k thật -> F thật
 0 |              — |        — |     33 |    33 | [-5, 10, -3, 7, 2] -> 33
 1 |              2 |        1 |     34 |    34 | [2, -5, 10, -3, 7] -> 34
 2 |              7 |      -24 |     10 |    34 | [7, 2, -5, 10, -3] -> 10
 3 |             -3 |       26 |     36 |    36 | [-3, 7, 2, -5, 10] -> 36
 4 |             10 |      -39 |     -3 |    36 | [10, -3, 7, 2, -5] -> -3
kết quả: 36


## 9. Code nộp LeetCode

Lời giải chính thức của notebook là **Cách 3** (`O(n)` thời gian, `O(1)` bộ nhớ).

In [20]:
class Solution:
    def maxRotateFunction(self, nums: List[int]) -> int:
        n, s = len(nums), sum(nums)
        f = sum(i * v for i, v in enumerate(nums))      # F(0)
        ans = f
        for k in range(1, n):
            f += s - n * nums[n - k]                    # F(k) = F(k-1) + S - n * nums[n-k]
            ans = max(ans, f)
        return ans


for nums, expected in examples:
    got = Solution().maxRotateFunction(nums)
    print(nums, "->", got)
    assert got == expected

[4, 3, 2, 6] -> 26
[100] -> 0


## 10. Vì sao đúng

1. **Công thức truy hồi đúng:** đã chứng minh ở mục 6: xoay một bước làm phần tử cuối mất `(n-1)*last` và `n-1` phần tử còn lại mỗi phần tử được `+1` lần giá trị, tổng `S - last`. Hiệu: `S - last - (n-1)*last = S - n*last`.
2. **Phần tử cuối đúng là `nums[n-k]`:** `arr_{k-1}[n-1] = nums[(n-1-(k-1)) mod n] = nums[n-k]` với `1 <= k <= n-1` (không cần `mod` vì `n-k` nằm trong `[1, n-1]`).
3. **Đủ mọi phép xoay:** `k` chạy `0..n-1` — xoay `n` bước quay về `arr_0`, nên không cần xét thêm.
4. **Quy nạp:** `f` khởi tạo bằng `F(0)` (tính trực tiếp, đúng). Nếu `f = F(k-1)` thì sau cập nhật `f = F(k)`. `ans` là max của mọi `F` đã duyệt.

In [21]:
# Kiểm chứng bằng code: F(k) - F(k-1) == S - n * nums[n-k] với MỌI k trên nhiều input ngẫu nhiên
random.seed(10)
for _ in range(500):
    n = random.randint(2, 9)
    nums = [random.randint(-100, 100) for _ in range(n)]
    S = sum(nums)
    F = [sum(i * v for i, v in enumerate(nums[n - k:] + nums[:n - k])) for k in range(n)]
    for k in range(1, n):
        assert F[k] - F[k - 1] == S - n * nums[n - k]
print("OK: đẳng thức truy hồi đúng cho mọi k")

OK: đẳng thức truy hồi đúng cho mọi k


## 11. Time / Space Complexity

| Bước | Chi phí |
|---|---|
| `sum(nums)` | `O(n)` |
| Tính `F(0)` | `O(n)` |
| `n-1` lần cập nhật | `O(n)` (mỗi lần `O(1)`) |
| **Tổng** | **Time `O(n)`, Space `O(1)`** |

Giá trị: `|F(k)| <= (n-1)*n/2 * 100 ≈ 5*10^11` với `n = 10^5` — Python không tràn số; ở Java/C++ cần dùng `long` cho các biến trung gian (dù đáp án cuối vừa 32 bit).

## 12. Bộ test

Dùng `max_rotate_brute` (Cách 1, bám sát đề) làm **oracle** độc lập. Các test: ví dụ đề, biên, phản ví dụ cho các cách sai, đối chiếu bốn cách, ngẫu nhiên có seed đúng ràng buộc, và hiệu năng ở `n = 10^5`.

In [22]:
# 12.1 Ví dụ của đề + biên
cases = [
    ([4, 3, 2, 6], 26),
    ([100], 0),
    ([1, 2], 2),                      # F(0)=2, F(1)=1
    ([2, 1], 2),
    ([0, 0, 0], 0),                   # toàn 0
    ([5, 5, 5, 5], 30),               # mọi F bằng 5*(0+1+2+3)
    ([-1, -2], -1),                   # đáp án âm
    ([-100] * 5, -1000),              # toàn âm bằng nhau
    ([1, 2, 3], 8),
]
for nums, expected in cases:
    for sol in (Solution(), SolutionB(), SolutionC(), SolutionD()):
        got = sol.maxRotateFunction(list(nums))
        assert got == expected, (type(sol).__name__, nums, got, expected)
    assert max_rotate_brute(nums) == expected
print("OK: ví dụ đề + biên cho cả 4 cách và oracle")

OK: ví dụ đề + biên cho cả 4 cách và oracle


In [23]:
# 12.2 Phản ví dụ: nhiều F khác nhau, max không ở k = 0 và cũng không ở k = n-1
nums = [1, -2, 3, -4, 5]
n = len(nums)
Fs = [sum(i * v for i, v in enumerate(nums[n - k:] + nums[:n - k])) for k in range(n)]
print("F =", Fs, "| max =", max(Fs), "tại k =", Fs.index(max(Fs)))
assert Solution().maxRotateFunction(nums) == max(Fs)

F = [12, -10, 13, 1, 14] | max = 14 tại k = 4


In [24]:
# 12.3 Đối chiếu bốn cách + oracle trên nhiều input ngẫu nhiên nhỏ (seed cố định, đúng ràng buộc)
random.seed(396)
for _ in range(2000):
    n = random.randint(1, 12)
    nums = [random.randint(-100, 100) for _ in range(n)]
    expect = max_rotate_brute(nums)
    assert Solution().maxRotateFunction(nums) == expect, nums
    assert SolutionB().maxRotateFunction(nums) == expect, nums
    assert SolutionC().maxRotateFunction(nums) == expect, nums
    assert SolutionD().maxRotateFunction(nums) == expect, nums
    assert max_rotate_index(nums) == max_rotate_recurrence(nums) == max_rotate_closed(nums) == expect
print("OK: 2000 input ngẫu nhiên, bốn cách khớp oracle")

OK: 2000 input ngẫu nhiên, bốn cách khớp oracle


In [25]:
# 12.4 Tính chất: hằng số thêm vào mọi phần tử làm F(k) dịch đúng c * n(n-1)/2 (nên argmax không đổi)
random.seed(7)
for _ in range(300):
    n = random.randint(1, 10)
    nums = [random.randint(-50, 50) for _ in range(n)]
    c = random.randint(-30, 30)
    shifted = [v + c for v in nums]
    assert Solution().maxRotateFunction(shifted) == Solution().maxRotateFunction(nums) + c * n * (n - 1) // 2
print("OK: tính chất dịch hằng số")

OK: tính chất dịch hằng số


In [26]:
# 12.5 Hiệu năng: n = 10^5 (ràng buộc tối đa), chỉ chạy Cách 3 và 4 (Cách 1, 2 sẽ rất chậm)
random.seed(100000)
big = [random.randint(-100, 100) for _ in range(10**5)]
t = time.perf_counter()
r3 = Solution().maxRotateFunction(big)
elapsed = time.perf_counter() - t
r4 = SolutionD().maxRotateFunction(big)
assert r3 == r4
assert elapsed < 2.0
print("n = 10^5 : kết quả", r3, "| Cách 3 chạy xong dưới 2 giây:", elapsed < 2.0)

n = 10^5 : kết quả 307120438 | Cách 3 chạy xong dưới 2 giây: True


In [27]:
# 12.6 So sánh tốc độ Cách 2 (O(n^2)) với Cách 3 (O(n)) ở n = 1500 để thấy khác biệt bậc
random.seed(1500)
mid = [random.randint(-100, 100) for _ in range(1500)]
t0 = time.perf_counter(); a = SolutionB().maxRotateFunction(mid); t_b = time.perf_counter() - t0
t0 = time.perf_counter(); b = SolutionC().maxRotateFunction(mid); t_c = time.perf_counter() - t0
assert a == b
print("kết quả giống nhau; Cách 3 nhanh hơn Cách 2 ít nhất 20 lần:", t_c * 20 < t_b)

kết quả giống nhau; Cách 3 nhanh hơn Cách 2 ít nhất 20 lần: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: khởi tạo `ans = 0`

Đáp án có thể **âm** (mọi `F(k)` đều âm). Khởi tạo `0` sẽ vô tình "thắng" mọi giá trị âm.

In [28]:
def wrong_init_zero(nums):
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    ans = 0                                   # SAI: nên là f (F(0))
    for k in range(1, n):
        f += s - n * nums[n - k]
        ans = max(ans, f)
    return ans


print("sai:", wrong_init_zero([-1, -2]), "| đúng:", Solution().maxRotateFunction([-1, -2]))
assert wrong_init_zero([-1, -2]) != Solution().maxRotateFunction([-1, -2])

sai: 0 | đúng: -1


**Vì sao sai:** với `[-1, -2]`, `F(0) = -2`, `F(1) = -1`, đáp án đúng `-1`; code sai trả `0`. **Cách sửa:** `ans = f` ngay sau khi tính `F(0)`.

### Lỗi 2: dùng nhầm chỉ số `nums[i]` thay vì `nums[n - i]`

Phần tử bị đưa từ cuối lên đầu ở bước `k` là `nums[n - k]`, **không** phải `nums[k]` hay `nums[k-1]`.

In [29]:
def wrong_index(nums):
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    ans = f
    for k in range(1, n):
        f += s - n * nums[k]                  # SAI: phải là nums[n - k]
        ans = max(ans, f)
    return ans


print("sai:", wrong_index([4, 3, 2, 6]), "| đúng:", Solution().maxRotateFunction([4, 3, 2, 6]))
assert wrong_index([4, 3, 2, 6]) != 26

sai: 35 | đúng: 26


**Vì sao sai:** bước `k = 1` phải trừ `n * 6` (phần tử cuối `6`), nhưng code sai trừ `n * nums[1] = n * 3`. **Cách sửa:** viết ra `arr_{k-1}[-1]` bằng tay một lần để chắc chắn nó là `nums[n-k]`.

### Lỗi 3: quên cộng dồn `f` (tính lại từ `F(0)` mỗi bước)

In [30]:
def wrong_no_accumulate(nums):
    n, s = len(nums), sum(nums)
    f0 = sum(i * v for i, v in enumerate(nums))
    ans = f0
    for k in range(1, n):
        f = f0 + s - n * nums[n - k]          # SAI: phải dựa trên F(k-1), không phải F(0)
        ans = max(ans, f)
    return ans


print("sai:", wrong_no_accumulate([4, 3, 2, 6]), "| đúng:", Solution().maxRotateFunction([4, 3, 2, 6]))
assert wrong_no_accumulate([4, 3, 2, 6]) != 26

sai: 32 | đúng: 26


**Vì sao sai:** công thức là `F(k) = F(k-1) + ...`; dùng `F(0)` cố định chỉ đúng ở `k = 1`. Từ `k = 2` giá trị sai (`25 + 15 - 8 = 32`). **Cách sửa:** `f += s - n * nums[n - k]` (cộng vào chính `f`).

### Lỗi 4: nhầm hệ số `n` với `n - 1`

In [31]:
def wrong_coefficient(nums):
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    ans = f
    for k in range(1, n):
        f += s - (n - 1) * nums[n - k]        # SAI: phần tử cuối cũng nằm trong S, nên phải là n * last
        ans = max(ans, f)
    return ans


print("sai:", wrong_coefficient([4, 3, 2, 6]), "| đúng:", Solution().maxRotateFunction([4, 3, 2, 6]))
assert wrong_coefficient([4, 3, 2, 6]) != 26

sai: 37 | đúng: 26


**Vì sao sai:** `S` **đã gồm** `last`. Hiệu đúng là `(S - last) - (n-1)*last = S - n*last`. Dùng `(n-1)` nghĩa là quên rằng `last` bị tính một lần thừa trong `S`. **Cách sửa:** luôn kiểm tra công thức bằng ví dụ nhỏ (như bảng ở mục 6).

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "xoay mảng" + "tổng có trọng số theo vị trí" ⇒ nghĩ ngay tới việc tìm **hiệu giữa hai trạng thái liền kề** thay vì tính lại. Đây là kỹ thuật **rolling / sliding update** (cùng họ với cửa sổ trượt, hash lăn).
- **Câu hỏi vàng:** "Khi chuyển từ trạng thái `k-1` sang `k`, cái gì thay đổi?" — ở đây chỉ **một** phần tử đổi hệ số lớn, còn lại cùng đổi `+1`.
- **Hệ số tăng đều `+1` cho cả nhóm** ⇒ gom lại thành **tổng nhóm** (`S - last`) thay vì cộng từng phần tử.
- **Khi đáp án có thể âm**, khởi tạo `ans` bằng một giá trị thật của bài (không phải `0`).
- **Kiểm tra nhanh** công thức mới: so với brute force trên 1000 input nhỏ ngẫu nhiên (như mục 12.3).

## 15. Biến thể

1. **Trả về `k` đạt giá trị lớn nhất** (nhỏ nhất nếu có nhiều `k`).
2. **Giá trị nhỏ nhất** của `F`: chỉ đổi `max` thành `min`.
3. **Xoay ngược chiều kim đồng hồ:** tập các mảng xoay không đổi, nên max không đổi.
4. **Hệ số bắt đầu từ 1** (`1*arr[0] + 2*arr[1] + ...`): mọi `F(k)` cộng thêm đúng `S` ⇒ đáp án cộng `S`.

In [32]:
def argmax_rotation(nums):
    """Trả về (k nhỏ nhất đạt max, giá trị max)."""
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    best, best_k = f, 0
    for k in range(1, n):
        f += s - n * nums[n - k]
        if f > best:
            best, best_k = f, k
    return best_k, best


def min_rotate_function(nums):
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    ans = f
    for k in range(1, n):
        f += s - n * nums[n - k]
        ans = min(ans, f)
    return ans


print(argmax_rotation([4, 3, 2, 6]))          # (3, 26)
print(min_rotate_function([4, 3, 2, 6]))      # 16
# biến thể hệ số bắt đầu từ 1: đáp án = đáp án gốc + sum(nums)
print(Solution().maxRotateFunction([4, 3, 2, 6]) + sum([4, 3, 2, 6]))   # 41

(3, 26)
16
41


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute Force | dựng từng `arr_k`, tính `F` | `O(n²)` | `O(n)` | hiểu đề, làm **oracle** kiểm tra |
| 2. Công thức chỉ số | `nums[(i-k) % n]`, không dựng mảng | `O(n²)` | `O(1)` | `n` nhỏ, tiết kiệm bộ nhớ |
| 3. Truy hồi | `F(k)=F(k-1)+S-n*nums[n-k]` | `O(n)` | `O(1)` | **nộp bài** (nhanh, gọn) |
| 4. Công thức đóng | `F(k)=F(0)+k*S-n*suffix(k)` | `O(n)` | `O(n)` | cần `F(k)` tại `k` bất kỳ (truy vấn) |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `[4, 3, 2, 6][-1:] + [4, 3, 2, 6][:-1]`; (b) `(-3) % 5`; (c) `sum(i * v for i, v in enumerate([5, 1]))`; (d) với `nums = [1, 2]`, `S - n * nums[n - 1]`.

**Bài 2 (tính tay).** Với `nums = [1, 2, 3]`, tính `F(0), F(1), F(2)` bằng hai cách (dựng mảng xoay và công thức truy hồi) rồi chọn max.

**Bài 3 (điền TODO).** Hoàn thành `rotate_todo(nums)` theo Cách 3.

In [33]:
def rotate_todo(nums):
    # TODO 1: n = len(nums), s = sum(nums)
    # TODO 2: f = F(0) bằng sum(i * v for i, v in enumerate(nums)); ans = f
    # TODO 3: for k in range(1, n): cập nhật f += s - n * nums[n - k]
    # TODO 4: cập nhật ans = max(ans, f) và trả về ans
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `rotate_closed(nums)` theo Cách 4 (tổng tiền tố) **không nhìn lại** code ở mục 7; kiểm tra với `[4, 3, 2, 6]` (kết quả `26`).

**Bài 5 (biến thể / mini challenge).** Viết `count_best(nums)` trả về **số giá trị `k`** trong `0..n-1` cho `F(k)` bằng giá trị lớn nhất. Kiểm tra: `[5, 5, 5, 5] → 4`, `[4, 3, 2, 6] → 1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[6, 4, 3, 2]`; `2`; `1`; `3 - 2*2 = -1`. Bài 3: nhìn lại hàm `max_rotate_recurrence` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `F(0) = 0+2+6 = 8`; `S = 6`; `F(1) = 8 + 6 - 3*3 = 5`; `F(2) = 5 + 6 - 3*2 = 5`. Bài 4: `F(k) = F(0) + k*S - n*(prefix[n] - prefix[n-k])`.
- **Mức 3 (sâu):** Bài 5: tính cả danh sách `F(k)` bằng truy hồi rồi đếm số lần xuất hiện của `max(F)`; để tiết kiệm bộ nhớ, dùng hai biến `best` và `count`, reset `count = 1` khi gặp max mới, `count += 1` khi bằng max.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert [4, 3, 2, 6][-1:] + [4, 3, 2, 6][:-1] == [6, 4, 3, 2]
assert (-3) % 5 == 2
assert sum(i * v for i, v in enumerate([5, 1])) == 1
assert sum([1, 2]) - 2 * [1, 2][1] == -1

# Bài 2
nums2 = [1, 2, 3]
n2, s2 = 3, 6
F2 = [8]
for k in range(1, n2):
    F2.append(F2[-1] + s2 - n2 * nums2[n2 - k])
assert F2 == [8, 5, 5]
assert F2 == [sum(i * v for i, v in enumerate(nums2[n2 - k:] + nums2[:n2 - k])) for k in range(n2)]
assert max(F2) == 8


# Bài 3
def rotate_todo_ans(nums):
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    ans = f
    for k in range(1, n):
        f += s - n * nums[n - k]
        ans = max(ans, f)
    return ans


for nums, expected in examples:
    assert rotate_todo_ans(nums) == expected


# Bài 4
def rotate_closed(nums):
    n = len(nums)
    prefix = [0] + list(accumulate(nums))
    f0 = sum(i * v for i, v in enumerate(nums))
    return max(f0 + k * prefix[n] - n * (prefix[n] - prefix[n - k]) for k in range(n))


assert rotate_closed([4, 3, 2, 6]) == 26 and rotate_closed([100]) == 0


# Bài 5
def count_best(nums):
    n, s = len(nums), sum(nums)
    f = sum(i * v for i, v in enumerate(nums))
    best, count = f, 1
    for k in range(1, n):
        f += s - n * nums[n - k]
        if f > best:
            best, count = f, 1
        elif f == best:
            count += 1
    return count


assert count_best([5, 5, 5, 5]) == 4
assert count_best([4, 3, 2, 6]) == 1
random.seed(5)
for _ in range(300):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 8))]
    n = len(nums)
    Fs = [sum(i * v for i, v in enumerate(nums[n - k:] + nums[:n - k])) for k in range(n)]
    assert count_best(nums) == Fs.count(max(Fs))
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (truy hồi một biến)

```python
class Solution:
    def maxRotateFunction(self, nums: List[int]) -> int:
        n, s = len(nums), sum(nums)
        f = sum(i * v for i, v in enumerate(nums))   # F(0)
        ans = f
        for k in range(1, n):
            f += s - n * nums[n - k]                 # F(k) = F(k-1) + S - n * nums[n-k]
            ans = max(ans, f)
        return ans
```

**Checklist**

- [ ] Đã khởi tạo `ans = F(0)` (không phải `0`) vì đáp án có thể âm?
- [ ] Phần tử cuối của `arr_{k-1}` là `nums[n - k]` (không phải `nums[k]`)?
- [ ] `f` được **cộng dồn** (`f += ...`), không tính lại từ `F(0)`?
- [ ] Hệ số là `n * last` (vì `S` đã chứa `last`)?
- [ ] Đã so sánh với brute force trên input nhỏ ngẫu nhiên và thử `n = 1`, toàn âm, toàn bằng nhau?
- [ ] Với Java/C++: dùng `long` cho `f` và `S`?